In [1]:
from __future__ import annotations
import os
import sys
from pathlib import Path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from utils.config import MainConfig
from arc.model import KilatTransformer
from training.trainer import KilatTrainer, TrainingArguments
from data.collator import KilatDataCollator
from data.dataset import KilatDataset
from torchinfo import summary
import torch
import json
from torch.utils.data import DataLoader
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.compute as pc

# Use `config.yaml` To Make Your Life Easier

Instead of wrapping the config with `build_config` from `config.py`, you can use a YAML file directly.

**Why YAML is better:**

```python
# Old way: Hardcoded configs scattered in Python
config = KilatConfig(
    vocab_size=32000,
    n_embd=768,
    n_head=12,
    n_layer=12,
    ffn_mode="dense"
)

# Or worse: Wrapped with build_config boilerplate
config = build_config(
    model_size="small",
    use_moe=False,
    batch_size=32
)

# Better: Clean YAML configuration
import yaml

with open("config.yaml", "r") as f:
    config_dict = yaml.safe_load(f)
    config = KilatConfig(**config_dict)
```

**Example `config.yaml`:**
```yaml
# Model architecture
vocab_size: 32000
n_embd: 768
n_head: 12
n_layer: 12
ffn_mode: "dense"

# Training settings
batch_size: 32
learning_rate: 3e-4
warmup_steps: 2000

# System
dtype: "bfloat16"
gradient_checkpointing: true
```

**Benefits:**
- **Separation of concerns** - Code vs configuration
- **Easy to version control** - Human-readable diffs
- **Experiment tracking** - Just copy the YAML file
- **No code changes** to tweak hyperparameters
- **Multi-environment support** - dev.yaml, prod.yaml, benchmark.yaml

**Quick tip:** Use Pydantic or dataclasses to validate YAML at load time:
```python
class KilatConfig:
    @classmethod
    def from_yaml(cls, path: str):
        with open(path) as f:
            data = yaml.safe_load(f)
        return cls(**data)  # Validation happens here
```

---

This keeps your code clean and your experiments reproducible! 

In [2]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
config = MainConfig.from_yaml("../configs/small_dense.yaml")
model = KilatTransformer(config.model)
model = model.to(device)

# Model Architecture Analysis: KilatTransformer

This document provides a detailed breakdown of the `KilatTransformer` model based on its structural summary, parameter distribution, and computational complexity.

## 1. Executive Summary

`KilatTransformer` is a lightweight, GPT-style autoregressive transformer model designed for language modeling or sequence generation.

* **Total Parameters:** ~143.87 Million (143,865,672), placing it in the same tier as **GPT-2 Small** (~124M) or **BERT-Base** (~110M).
* **Vocabulary Size:** 32,000 tokens (standard for modern tokenizers like LLaMA or Byte-Pair Encoding).
* **Sequence Length (Context Window):** 512 tokens.
* **Hidden Dimension ($d_{model}$):** 768.
* **Number of Layers (Blocks):** 12 blocks.

---

## 2. Structural Breakdown

The model follows a clear three-stage pipeline: **Embedding**, **Transformer Backbone**, and **Language Model (LM) Head**.

```
[Input: 1 x 512] 
       │
       ▼
 [Embedding] ──► [Dropout]
       │
       ▼
 [12x Transformer Blocks (KilatAttention + FeedForward)]
       │
       ▼
 [RMSNorm] ──► [Linear LM Head] ──► [Output: 1 x 512 x 32000]

```

### A. Input & Embedding Layer (`1-1` to `1-2`)

* **Input Shape:** `[1, 512]` representing `[Batch Size, Sequence Length]`.
* **Embedding Matrix:** Maps token IDs to a dense space of size `768`.
* **Weight Tying Indicator:** The embedding layer has `24,576,000` parameters ($32,000 \times 768$). The final output `Linear` head also has exactly `24,576,000` parameters. This strongly implies **Weight Tying** (sharing weights between embedding and final linear projection) is active to save memory.

### B. The Transformer Backbone (`1-3`)

The core features a `ModuleList` containing **12 identical structural blocks** (`Block: 2-1` to `Block: 2-12`). Each block contains:

* **Pre-Normalization (`RMSNorm`):** Uses Root Mean Square Normalization rather than traditional LayerNorm, aligning with modern architectures like LLaMA to achieve faster training stability.
* **KilatAttention:** A custom attention mechanism utilizing ~3.17M parameters per layer.
* **FeedForward Network (FFN):** A position-wise feed-forward network handling ~4.72M parameters per layer. Given the hidden dimension of 768, the intermediate expansion dimension is likely 3072 ($4 \times 768$).

### C. Output Head (`1-4` to `1-5`)

* **Final Norm:** A closing `RMSNorm` cleans up the residual streams before projection.
* **Linear Projection:** Projects the hidden state back from `768` to the vocabulary size of `32,000` to yield token probabilities.

---

## 3. Parameter and Computational Cost Breakdown

Below is a granular view of where the parameters and Multiply-Accumulate (MAC) operations are allocated within a single forward pass:

| Sub-Module / Layer | Parameters (Per Instance) | Count | Total Parameters | % of Model |
| --- | --- | --- | --- | --- |
| **Token Embedding** | 24,576,000 | 1 | 24,576,000 | 17.08% |
| **RMSNorm** | 768 | 25 | 19,200 | 0.01% |
| **KilatAttention** | 3,172,614 | 12 | 38,071,368 | 26.46% |
| **FeedForward (FFN)** | 4,718,592 | 12 | 56,623,104 | 39.36% |
| **LM Head (Linear)** | 24,576,000 | 1 | 24,576,000 | 17.08% |
| **Total** |  |  | **143,865,672** | **100%** |


> 💡 **Observation:** The FeedForward networks represent the heaviest computational footprints inside the backbone layers ($39.36\%$), which is typical for standard Transformer setups.

---

## 4. Hardware & Memory Footprint Estimates

* **Parameters Size (575.46 MB):** This indicates the model weights are likely loaded/stored in **FP32 (32-bit floating-point)** precision ($143.8M \times 4\text{ bytes} \approx 575.4\text{ MB}$). If converted to FP16 or BF16, this size cuts down to **~288 MB**.
* **Forward/Backward Pass Size (678.43 MB):** Represents the memory allocated to store intermediate activations required to compute gradients during training.
* **Estimated Total Memory Required (1.25 GB):** Minimum VRAM buffer required for a single-batch training footprint. For batched training, scale this value proportionally alongside optimizer state multipliers (e.g., AdamW adds $8\text{ bytes}$ per parameter).

In [3]:
stats = summary(
    model,
    input_data={
        'input_ids': torch.randint(0, 32000, (1, 512), dtype=torch.long, device=device)
    },
    device=device, 
    depth=3,
    col_names=["input_size", "output_size", "num_params", "mult_adds"],
    verbose=1
)

Layer (type:depth-idx)                   Input Shape               Output Shape              Param #                   Mult-Adds
KilatTransformer                         --                        [1, 512, 32000]           --                        --
├─Embedding: 1-1                         [1, 512]                  [1, 512, 768]             24,576,000                24,576,000
├─Dropout: 1-2                           [1, 512, 768]             [1, 512, 768]             --                        --
├─ModuleList: 1-3                        --                        --                        --                        --
│    └─Block: 2-1                        [1, 512, 768]             [1, 512, 768]             --                        --
│    │    └─RMSNorm: 3-1                 [1, 512, 768]             [1, 512, 768]             768                       768
│    │    └─KilatAttention: 3-2          [1, 512, 768]             [1, 512, 768]             3,172,614                 3,172,608
│

# Quickstart: How to Use KilatDataset

This guide demonstrates how to load and use `KilatDataset` across different file formats.

> 💡 **Note:** For demonstration purposes, this guide uses a small **dummy dataset** created on the fly. In a production or large-scale training environment, you would simply replace the dummy file paths with the paths to your **actual training datasets** (e.g., a 100GB Parquet data lake or a massive JSONL corpus). The library mechanics remain exactly the same.

In [4]:
def create_dummy_data():
    return [
        {"input_ids": [101, 2054, 2003, 1037, 2771, 102], "text_meta": "Sample 1"},
        {"input_ids": [101, 1045, 2307, 2022, 3632, 102], "text_meta": "Sample 2"},
        {"input_ids": [101, 2044, 2006, 2311, 2019, 102], "text_meta": "Sample 3"},
        {"input_ids": [101, 1996, 2111, 2000, 2961, 102], "text_meta": "Sample 4"},
    ]

In [5]:
raw_data = create_dummy_data()
parquet_path = "demo_data.parquet"
jsonl_path = "demo_data.jsonl"
json_path = "demo_data.json"

In [6]:
# Save as Parquet and JSONL for demonstration
table = pa.Table.from_pylist(raw_data)
pq.write_table(table, parquet_path)
with open(jsonl_path, "w", encoding="utf-8") as f:
        for item in raw_data:
            f.write(json.dumps(item) + "\n")

In [7]:
# Initialize dataset pointing to your actual Parquet file
# It only loads the "input_ids" column, ignoring any heavy metadata columns automatically.
dataset = KilatDataset(file_or_data=parquet_path, key_name="input_ids")

print(f"Total rows in actual dataset: {len(dataset)}")
print(f"First sample: {dataset[0]}")  # Output: {"input_ids": [...]}

Total rows in actual dataset: 4
First sample: {'input_ids': [101, 2054, 2003, 1037, 2771, 102]}


# DataLoader Demonstration for Dataset Inspection

## What This Code Does

| Component | Purpose |
|-----------|---------|
| `PAD_TOKEN = 0` | Token ID used for padding sequences to equal length |
| `BLOCK_SIZE = 512` | Maximum sequence length (truncates longer sequences) |
| `IGNORE_INDEX = -100` | Value ignored in loss calculation (typically for padding positions) |
| `KilatDataCollator` | Custom collator that pads batches dynamically |
| `DataLoader` | Iterates through dataset, creates batches of size 2 |

## Important Note

> ⚠️ **You don't need to manually create this DataLoader when using `KilatTrainer`**

The `KilatTrainer` class **automatically creates its own DataLoader** internally based on the `TrainingArguments` you provide:

```python
# The trainer does this automatically in __init__:
self.train_dataloader = DataLoader(
    self.train_dataset,
    batch_size=self.args.per_device_train_batch_size,
    shuffle=True,
    collate_fn=self.data_collator,
    pin_memory=torch.cuda.is_available(),
)
```

## Why Show This Code?

This code is shown **only for demonstration purposes** to:

1. **Inspect the dataset structure** - See what a single batch looks like
2. **Verify tokenization** - Check if token IDs are correct before training
3. **Debug collation logic** - Ensure padding and truncation work as expected
4. **Understand data flow** - Visualize how data moves from dataset to model


> **For actual training, skip this manual DataLoader creation.** Just pass your `dataset` directly to `KilatTrainer`, and let the trainer handle DataLoader construction. This code snippet is purely for **data inspection and debugging** before you launch full training.

```python
# What you actually do for training:
trainer = KilatTrainer(
    model=model,
    args=args,
    train_dataset=dataset,        # ← Your dataset, NOT a DataLoader
    eval_dataset=eval_dataset,
    data_collator=collate_fn,     # ← Your collate function
)
trainer.train()  # Trainer creates DataLoader internally
```

In [8]:
PAD_TOKEN = 0     
BLOCK_SIZE = 512    
IGNORE_INDEX = -100

collate_fn = KilatDataCollator(
        pad_token_id=PAD_TOKEN,
        max_length=BLOCK_SIZE,
        ignore_index=-100,
    )


data_loader = DataLoader(
    dataset, 
    batch_size=2, 
    shuffle=True, 
    collate_fn=collate_fn
)

In [9]:
for step, batch in enumerate(data_loader):
    print(f"\n[ BATCH #{step + 1} ]")
    print("-" * 30)
    
    # 1. Check shape/dimensions of the generated Tensor
    print(f"Input IDs Shape : {batch['input_ids'].shape}  -> [Batch Size, Sequence Length]")
    print(f"Labels Shape    : {batch['labels'].shape}  -> [Batch Size, Sequence Length]")
    print("-" * 30)
    
    # 2. Print Input IDs Tensor data
    print("Input IDs Tensor (Data fed into the model):")
    print(batch['input_ids'])
    print()
    
    # 3. Print Labels Tensor data
    print("Labels Tensor (Target for Loss Calculation):")
    print(batch['labels'])
    print("=" * 50)

print("=== BATCH DEMONSTRATION COMPLETE ===")


[ BATCH #1 ]
------------------------------
Input IDs Shape : torch.Size([2, 6])  -> [Batch Size, Sequence Length]
Labels Shape    : torch.Size([2, 6])  -> [Batch Size, Sequence Length]
------------------------------
Input IDs Tensor (Data fed into the model):
tensor([[ 101, 1045, 2307, 2022, 3632,  102],
        [ 101, 2044, 2006, 2311, 2019,  102]])

Labels Tensor (Target for Loss Calculation):
tensor([[ 101, 1045, 2307, 2022, 3632,  102],
        [ 101, 2044, 2006, 2311, 2019,  102]])

[ BATCH #2 ]
------------------------------
Input IDs Shape : torch.Size([2, 6])  -> [Batch Size, Sequence Length]
Labels Shape    : torch.Size([2, 6])  -> [Batch Size, Sequence Length]
------------------------------
Input IDs Tensor (Data fed into the model):
tensor([[ 101, 2054, 2003, 1037, 2771,  102],
        [ 101, 1996, 2111, 2000, 2961,  102]])

Labels Tensor (Target for Loss Calculation):
tensor([[ 101, 2054, 2003, 1037, 2771,  102],
        [ 101, 1996, 2111, 2000, 2961,  102]])
=== BATCH DE

# Training

In [10]:
train_args = TrainingArguments(**config.training.to_dict())
# optional: save the config for reproducibility
# config.save_pretrained("./checkpoints/my-model")

In [13]:
trainer = KilatTrainer(
        model=model,
        args=train_args,
        train_dataset=dataset,
        # if you have eval you may use this
        # eval_dataset=eval_dataset,
        data_collator=collate_fn,
    )
trainer.train()


KilatTransformer Training
Output dir:          ./checkpoints/kilat-small-dense
Save checkpoints:    True
Training mode:       epochs
Total epochs:        5
Total target steps:  5
Batch size (per GPU):16
Gradient accum:      2
Effective batch:     32
Learning rate:       5.00e-05
Warmup steps:        500
Weight decay:        0.01
Max grad norm:       1.0
Device:              cuda
Precision:           FP16
GradScaler:          active
Reporting:           none
Seed:                42



Epoch 1/5: 100%|██████████| 1/1 [00:01<00:00,  1.18s/batch]



[Epoch 1] Complete.
Checkpoint saved: ./checkpoints/kilat-small-dense/checkpoint-epoch-1


Epoch 2/5: 100%|██████████| 1/1 [00:00<00:00,  3.47batch/s]



[Epoch 2] Complete.
Checkpoint saved: ./checkpoints/kilat-small-dense/checkpoint-epoch-2


...

Epoch 4/5: 100%|██████████| 1/1 [00:00<00:00,  9.60batch/s]


[Epoch 4] Complete.


Checkpoint saved: ./checkpoints/kilat-small-dense/checkpoint-epoch-4


Epoch 5/5: 100%|██████████| 1/1 [00:00<00:00,  3.77batch/s]



[Epoch 5] Complete.
Checkpoint saved: ./checkpoints/kilat-small-dense/checkpoint-epoch-5

Training complete (5 epochs, 0 steps)
Checkpoint saved: ./checkpoints/kilat-small-dense/checkpoint-final

Training Summary
Total steps:       0
Total time:        0.1 min (6.6 sec)
Avg steps/sec:     0.0
Best eval loss:    inf
Best eval PPL:     inf
Output directory:  ./checkpoints/kilat-small-dense

